# FSL-SAGE on Colab GPU

Clones this repo from GitHub, installs dependencies, and runs a quick GPU smoke test
(see `docs/part0-mnist-smoke-test.md` for the equivalent CPU run this mirrors).

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or better).

Datasets (`datas/`) and run outputs (`saves/`) are stored on your Google Drive so they
persist across Colab session resets instead of re-downloading/re-running every time.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Change BRANCH once this work lands on master.
REPO_URL = "https://github.com/juniorfelix998/FSL-SAGE.git"
BRANCH = "ft/add-mnist"

WORKSPACE = "/content/drive/MyDrive/fsl-sage-colab"
REPO_DIR = f"{WORKSPACE}/FSL-SAGE"

import os
os.makedirs(WORKSPACE, exist_ok=True)

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

In [ ]:
# src/main.py resolves datasets/saves as '../datas' and '../saves' relative to
# src/, so point those at persistent Drive folders instead of Colab's ephemeral disk.
DRIVE_DATAS = f"{WORKSPACE}/datas"
DRIVE_SAVES = f"{WORKSPACE}/saves"
os.makedirs(DRIVE_DATAS, exist_ok=True)
os.makedirs(DRIVE_SAVES, exist_ok=True)

for name, target in (("datas", DRIVE_DATAS), ("saves", DRIVE_SAVES)):
    link = f"{REPO_DIR}/{name}"
    if os.path.islink(link) or os.path.exists(link):
        continue
    os.symlink(target, link)

In [ ]:
# Same pinned pip list validated in docs/part0-mnist-smoke-test.md (the conda_env.yaml
# workaround for non-conda setups). nvidia-*-cu12/triton are intentionally omitted --
# Colab's CUDA runtime + the pip torch wheel pull those in automatically.
!pip install -q torch==2.5.1 torchvision==0.20.1 hydra-core==1.3.2 hydra-joblib-launcher==1.2.0 \
  omegaconf==2.3.0 wandb==0.19.3 numpy==2.1.3 pandas==2.2.3 scipy==1.14.1 matplotlib==3.9.2 \
  h5py==3.12.1 pyyaml==6.0.2 tqdm==4.67.0 requests==2.32.3 pillow==11.0.0 prettytable==3.12.0 \
  joblib==1.4.2 antlr4-python3-runtime==4.9.3 gitpython==3.1.43

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist

## Running a real experiment

Once the smoke test above completes, use the README's Hydra override syntax to run
any method/model/dataset combination, e.g.:

```python
!python main.py algorithm=fsl_sage model=resnet18 dataset=cifar10 \\
  dataset.distribution=noniid_dirichlet dataset.alpha=0.5 save=False device=cuda
```

Supported `algorithm=` keys: `fed_avg`, `sl_multi_server`, `sl_single_server`,
`cse_fsl`, `fsl_sage`. Results land under
`saves/<algorithm>/<model>/<dataset>-<distribution>/.../results.json`, which (via
the symlink above) is actually on your Drive at
`fsl-sage-colab/saves/...` so it survives runtime resets.